# FDA regulatory-data exploratory analysis

This notebook explores the PostgreSQL datasets used in the Qualifyze case study:

- inspections
- inspection citations
- compliance actions
- recalls
- published Form 483 records.

It is intentionally **read-only**. The goals are to understand table grain, coverage,
missingness, duplicates, temporal patterns, FEI overlap, target balance, and whether the
data is suitable for a point-in-time inspection-classification model.

## Set Up Configuration

In [ ]:
import os
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (
            (candidate / "config.yml").is_file()
            and (candidate / "pyproject.toml").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing "
        "config.yml and pyproject.toml"
    )


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

# Load configuration
from qualifyze.config import Settings

config = Settings()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sqlalchemy import URL, create_engine, text

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")

engine = create_engine(config.database.sqlalchemy_url, pool_pre_ping=True)

with engine.connect() as connection:
    connection.execute(text("SELECT 1"))

print(f"Connected to {config.database.name!r} on {config.database.host}:{config.database.port} as {config.database.username!r}")

## 2. Discover tables and resolve names

The project has used both singular and plural table names during development. This block
discovers the actual names rather than assuming them.

In [ ]:
def read_sql(sql: str, params: dict | None = None) -> pd.DataFrame:
    return pd.read_sql_query(text(sql), engine, params=params or {})


available_tables = read_sql(
    '''
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
      AND table_type = 'BASE TABLE'
    ORDER BY table_schema, table_name
    '''
)
display(available_tables)

TABLE_CANDIDATES = {
    "inspections": ("inspections", "inspection"),
    "inspections_citations": (
        "inspections_citations",
    ),
    "compliance_actions": (
        "compliance_actions",
        "compliance_action",
    ),
    "recalls": ("recalls", "recall"),
    "published483": ("published483", "published483s"),
}

discovered = {
    (row.table_schema, row.table_name)
    for row in available_tables.itertuples(index=False)
}

TABLES: dict[str, tuple[str, str]] = {}
for logical_name, candidates in TABLE_CANDIDATES.items():
    matches = [
        (schema, candidate)
        for schema, table_name in discovered
        for candidate in candidates
        if table_name == candidate
    ]
    if matches:
        TABLES[logical_name] = matches[0]

display(
    pd.DataFrame(
        [
            {
                "logical_name": logical,
                "schema": schema,
                "table_name": table,
            }
            for logical, (schema, table) in TABLES.items()
        ]
    )
)

missing_expected = sorted(set(TABLE_CANDIDATES) - set(TABLES))
if missing_expected:
    print("Expected datasets not found:", missing_expected)

In [ ]:
def quote_identifier(value: str) -> str:
    return '"' + value.replace('"', '""') + '"'


def qualified_table(logical_name: str) -> str:
    schema, table = TABLES[logical_name]
    return f"{quote_identifier(schema)}.{quote_identifier(table)}"


def table_columns(logical_name: str) -> list[str]:
    schema, table = TABLES[logical_name]
    result = read_sql(
        '''
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :schema
          AND table_name = :table
        ORDER BY ordinal_position
        ''',
        {"schema": schema, "table": table},
    )
    return result["column_name"].tolist()


def pick_column(logical_name: str, *candidates: str) -> str | None:
    available = set(table_columns(logical_name))
    return next((column for column in candidates if column in available), None)


def value_counts(
    logical_name: str,
    column: str,
    limit: int = 20,
) -> pd.DataFrame:
    column_sql = quote_identifier(column)
    return read_sql(
        f'''
        SELECT {column_sql} AS value, COUNT(*) AS row_count
        FROM {qualified_table(logical_name)}
        GROUP BY {column_sql}
        ORDER BY row_count DESC, value
        LIMIT :limit
        ''',
        {"limit": limit},
    )

## 3. Inventory, schemas, row counts, and data quality

In [ ]:
inventory_rows = []
for logical_name, (schema, table) in TABLES.items():
    row_count = int(
        read_sql(
            f"SELECT COUNT(*) AS n FROM {qualified_table(logical_name)}"
        ).loc[0, "n"]
    )
    inventory_rows.append(
        {
            "logical_name": logical_name,
            "schema": schema,
            "table_name": table,
            "row_count": row_count,
            "column_count": len(table_columns(logical_name)),
        }
    )

inventory = pd.DataFrame(inventory_rows).sort_values(
    "row_count", ascending=False
)
display(inventory)

In [ ]:
column_dictionary = read_sql(
    '''
    SELECT
        table_schema,
        table_name,
        ordinal_position,
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
    ORDER BY table_schema, table_name, ordinal_position
    '''
)
column_dictionary = column_dictionary[
    column_dictionary["table_name"].isin(
        [table for _, table in TABLES.values()]
    )
]
display(column_dictionary)

In [ ]:
def null_profile(logical_name: str) -> pd.DataFrame:
    columns = table_columns(logical_name)
    expressions = ["COUNT(*) AS total_rows"] + [
        (
            "COUNT(*) FILTER (WHERE "
            f"{quote_identifier(column)} IS NULL) "
            f"AS {quote_identifier(column)}"
        )
        for column in columns
    ]
    result = read_sql(
        f"SELECT {', '.join(expressions)} "
        f"FROM {qualified_table(logical_name)}"
    ).iloc[0]
    total = int(result["total_rows"])
    profile = pd.DataFrame(
        {
            "column": columns,
            "missing_count": [int(result[column]) for column in columns],
        }
    )
    profile["missing_rate"] = np.where(
        total > 0,
        profile["missing_count"] / total,
        np.nan,
    )
    return profile.sort_values(
        ["missing_rate", "column"], ascending=[False, True]
    )


for logical_name in TABLES:
    print(f"\n{logical_name}: missing-value profile")
    display(null_profile(logical_name).head(20))

In [ ]:
IDENTITY_KEYS = {
    "inspections": (
        "inspection_id",
        "project_area",
        "product_type",
        "additional_details",
    ),
    "inspection_citations": (
        "inspection_id",
        "program_area",
        "act_cfr_number",
        "short_description",
        "long_description",
    ),
    "compliance_actions": (
        "case_injunction_id",
        "fei_number",
        "product_type",
    ),
    "published483": ("record_id", "fei_number"),
    "recalls": ("fei_number", "event_id", "product_id"),
}


def duplicate_group_count(
    logical_name: str,
    requested_keys: tuple[str, ...],
) -> dict[str, object]:
    available = set(table_columns(logical_name))
    keys = [key for key in requested_keys if key in available]
    missing = sorted(set(requested_keys) - set(keys))
    if missing:
        return {
            "table": logical_name,
            "keys": ", ".join(keys),
            "duplicate_groups": np.nan,
            "note": f"Missing key columns: {missing}",
        }
    key_sql = ", ".join(quote_identifier(key) for key in keys)
    result = read_sql(
        f'''
        SELECT COUNT(*) AS duplicate_groups
        FROM (
            SELECT {key_sql}
            FROM {qualified_table(logical_name)}
            GROUP BY {key_sql}
            HAVING COUNT(*) > 1
        ) AS duplicated
        '''
    )
    return {
        "table": logical_name,
        "keys": ", ".join(keys),
        "duplicate_groups": int(result.loc[0, "duplicate_groups"]),
        "note": "",
    }


duplicate_summary = pd.DataFrame(
    [
        duplicate_group_count(logical_name, keys)
        for logical_name, keys in IDENTITY_KEYS.items()
        if logical_name in TABLES
    ]
)
display(duplicate_summary)

## 4. Inspections

FDA inspection data can contain multiple project-area rows for one physical inspection.
Always compare raw row counts with distinct `inspection_id` counts before modelling.

In [ ]:
if "inspections" in TABLES:
    inspection_columns = table_columns("inspections")
    inspection_id_col = pick_column("inspections", "inspection_id")
    fei_col = pick_column("inspections", "fei_number", "fei")
    classification_col = pick_column(
        "inspections", "classification_code", "classification"
    )
    inspection_date_col = pick_column(
        "inspections",
        "inspection_end_date",
        "end_date",
        "inspection_date",
    )

    print("Resolved inspection columns:")
    print(
        {
            "inspection_id": inspection_id_col,
            "fei": fei_col,
            "classification": classification_col,
            "date": inspection_date_col,
        }
    )

    if inspection_id_col and fei_col:
        overview = read_sql(
            f'''
            SELECT
                COUNT(*) AS raw_rows,
                COUNT(DISTINCT {quote_identifier(inspection_id_col)})
                    AS distinct_inspections,
                COUNT(DISTINCT {quote_identifier(fei_col)})
                    AS distinct_feis,
                COUNT(*) FILTER (
                    WHERE {quote_identifier(fei_col)} IS NULL
                       OR btrim({quote_identifier(fei_col)}::text) = ''
                ) AS rows_without_fei
            FROM {qualified_table("inspections")}
            '''
        )
        display(overview)

    for candidate in (
        classification_col,
        pick_column("inspections", "project_area"),
        pick_column("inspections", "product_type"),
        pick_column("inspections", "country_area", "country"),
    ):
        if candidate:
            print(f"\nTop values: {candidate}")
            display(value_counts("inspections", candidate, limit=20))
else:
    print("Inspection table not found.")

In [ ]:
if (
    "inspections" in TABLES
    and inspection_date_col
    and classification_col
):
    inspection_timeline = read_sql(
        f'''
        SELECT
            EXTRACT(YEAR FROM {quote_identifier(inspection_date_col)})::int
                AS year,
            {quote_identifier(classification_col)} AS classification,
            COUNT(*) AS row_count
        FROM {qualified_table("inspections")}
        WHERE {quote_identifier(inspection_date_col)} IS NOT NULL
          AND {quote_identifier(classification_col)} IS NOT NULL
        GROUP BY year, classification
        ORDER BY year, classification
        '''
    )
    display(inspection_timeline.tail(20))

    plt.figure(figsize=(12, 5))
    sns.lineplot(
        data=inspection_timeline,
        x="year",
        y="row_count",
        hue="classification",
        marker="o",
    )
    plt.title("Inspection-classification records by year")
    plt.ylabel("Rows (check project-area grain)")
    plt.tight_layout()
    plt.show()

In [ ]:
if "inspections" in TABLES and inspection_id_col and fei_col:
    inspections_per_fei = read_sql(
        f'''
        SELECT
            {quote_identifier(fei_col)} AS fei_number,
            COUNT(DISTINCT {quote_identifier(inspection_id_col)})
                AS inspection_count
        FROM {qualified_table("inspections")}
        WHERE {quote_identifier(fei_col)} IS NOT NULL
          AND btrim({quote_identifier(fei_col)}::text) <> ''
        GROUP BY {quote_identifier(fei_col)}
        '''
    )
    display(inspections_per_fei["inspection_count"].describe())
    print(
        "FEIs with at least two inspections:",
        int((inspections_per_fei["inspection_count"] >= 2).sum()),
    )

    plt.figure(figsize=(9, 4))
    sns.histplot(
        inspections_per_fei["inspection_count"].clip(upper=10),
        discrete=True,
    )
    plt.title("Distinct inspections per FEI (values above 10 clipped)")
    plt.xlabel("Inspection count")
    plt.tight_layout()
    plt.show()

## 5. Inspection citations and linkage coverage

In [ ]:
citation_table_key = "inspections_citations"
inspection_table_key = "inspections"

if citation_table_key not in TABLES:
    raise KeyError(
        f"{citation_table_key!r} is not in TABLES. "
        f"Available keys: {list(TABLES)}"
    )

if inspection_table_key not in TABLES:
    raise KeyError(
        f"{inspection_table_key!r} is not in TABLES. "
        f"Available keys: {list(TABLES)}"
    )

# Resolve the join columns independently so this cell does
# not depend on variables created in previous notebook cells.
citation_inspection_col = pick_column(
    citation_table_key,
    "inspection_id",
)

inspection_id_col = pick_column(
    inspection_table_key,
    "inspection_id",
)

if citation_inspection_col is None:
    raise KeyError(
        "Column 'inspection_id' was not found in "
        f"{qualified_table(citation_table_key)}"
    )

if inspection_id_col is None:
    raise KeyError(
        "Column 'inspection_id' was not found in "
        f"{qualified_table(inspection_table_key)}"
    )

print(
    "Citation table:",
    qualified_table(citation_table_key),
)
print(
    "Inspection table:",
    qualified_table(inspection_table_key),
)

# Explore the most relevant citation variables.
for candidate in (
    pick_column(
        citation_table_key,
        "program_area",
    ),
    pick_column(
        citation_table_key,
        "act_cfr_number",
        "cfr_number",
    ),
    pick_column(
        citation_table_key,
        "short_description",
    ),
):
    if candidate is not None:
        print(f"\nTop citation values: {candidate}")

        display(
            value_counts(
                citation_table_key,
                candidate,
                limit=20,
            )
        )

# Prepare safe SQL identifiers and resolved table names.
citation_table = qualified_table(
    citation_table_key
)

inspection_table = qualified_table(
    inspection_table_key
)

citation_inspection_id_sql = quote_identifier(
    citation_inspection_col
)

inspection_id_sql = quote_identifier(
    inspection_id_col
)

# Measure how many citation rows can be linked to an inspection.
citation_linkage = read_sql(
    f"""
    SELECT
        COUNT(*) AS citation_rows,

        COUNT(*) FILTER (
            WHERE inspected.{inspection_id_sql}
                IS NOT NULL
        ) AS linked_rows,

        COUNT(*) FILTER (
            WHERE inspected.{inspection_id_sql}
                IS NULL
        ) AS orphan_rows,

        COUNT(
            DISTINCT citation.{citation_inspection_id_sql}
        ) AS distinct_cited_inspections,

        COUNT(
            DISTINCT citation.fei_number
        ) AS distinct_cited_feis

    FROM {citation_table} AS citation

    LEFT JOIN (
        SELECT DISTINCT {inspection_id_sql}
        FROM {inspection_table}
    ) AS inspected
      ON inspected.{inspection_id_sql}
       = citation.{citation_inspection_id_sql}
    """
)

display(citation_linkage)

# Calculate the number of citations associated with each inspection.
citations_per_inspection = read_sql(
    f"""
    SELECT
        citation.{citation_inspection_id_sql}
            AS inspection_id,

        COUNT(*) AS citation_count,

        COUNT(
            DISTINCT citation.act_cfr_number
        ) AS distinct_cfr_count

    FROM {citation_table} AS citation

    GROUP BY
        citation.{citation_inspection_id_sql}

    ORDER BY
        citation_count DESC
    """
)

display(citations_per_inspection.head(20))

display(
    citations_per_inspection[
        [
            "citation_count",
            "distinct_cfr_count",
        ]
    ].describe()
)

# Plot the distribution. The upper tail is clipped only for
# visualization; the underlying data remains unchanged.
if not citations_per_inspection.empty:
    citation_plot_limit = max(
        1,
        int(
            citations_per_inspection[
                "citation_count"
            ].quantile(0.99)
        ),
    )

    plt.figure(figsize=(10, 5))

    sns.histplot(
        citations_per_inspection[
            "citation_count"
        ].clip(upper=citation_plot_limit),
        bins=min(citation_plot_limit, 30),
    )

    plt.title(
        "Citation count per inspection "
        "(values clipped at the 99th percentile)"
    )
    plt.xlabel("Citation count")
    plt.ylabel("Number of inspections")
    plt.tight_layout()
    plt.show()

## 6. Compliance actions

In [ ]:
if "compliance_actions" in TABLES:
    action_type_col = pick_column("compliance_actions", "action_type")
    action_date_col = pick_column(
        "compliance_actions",
        "action_taken_date",
        "action_date",
    )
    if action_type_col:
        display(value_counts("compliance_actions", action_type_col, 30))

    if action_type_col and action_date_col:
        action_timeline = read_sql(
            f'''
            SELECT
                EXTRACT(YEAR FROM {quote_identifier(action_date_col)})::int
                    AS year,
                {quote_identifier(action_type_col)} AS action_type,
                COUNT(*) AS row_count
            FROM {qualified_table("compliance_actions")}
            WHERE {quote_identifier(action_date_col)} IS NOT NULL
            GROUP BY year, action_type
            ORDER BY year, action_type
            '''
        )
        display(action_timeline.tail(30))
else:
    print("Compliance-action table not found.")

## 7. Recalls

In [ ]:
if "recalls" in TABLES:
    for candidate in (
        pick_column("recalls", "event_classification"),
        pick_column("recalls", "product_classification"),
        pick_column("recalls", "status"),
        pick_column("recalls", "center"),
        pick_column("recalls", "product_type"),
    ):
        if candidate:
            print(f"\nTop recall values: {candidate}")
            display(value_counts("recalls", candidate, 20))

    recall_date_col = pick_column(
        "recalls",
        "center_classification_date",
        "recall_initiation_date",
    )
    if recall_date_col:
        recall_timeline = read_sql(
            f'''
            SELECT
                EXTRACT(YEAR FROM {quote_identifier(recall_date_col)})::int
                    AS year,
                COUNT(DISTINCT event_id) AS recall_events,
                COUNT(*) AS product_rows
            FROM {qualified_table("recalls")}
            WHERE {quote_identifier(recall_date_col)} IS NOT NULL
            GROUP BY year
            ORDER BY year
            '''
        )
        display(recall_timeline.tail(20))
else:
    print("Recall table not found.")

## 8. Published Form 483 records

Use `publish_date` for information-availability checks. `record_date` describes the
underlying record, but a model could not have consumed that record publicly before it was
published.

In [ ]:
if "published483" in TABLES:
    for candidate in (
        pick_column("published483", "record_type"),
        pick_column("published483", "legal_name"),
    ):
        if candidate:
            print(f"\nTop Published 483 values: {candidate}")
            display(value_counts("published483", candidate, 20))

    record_date_col = pick_column("published483", "record_date")
    publish_date_col = pick_column(
        "published483", "publish_date", "published_date"
    )
    if record_date_col and publish_date_col:
        publication_lag = read_sql(
            f'''
            SELECT
                {quote_identifier(publish_date_col)}
                  - {quote_identifier(record_date_col)} AS publication_lag_days
            FROM {qualified_table("published483")}
            WHERE {quote_identifier(record_date_col)} IS NOT NULL
              AND {quote_identifier(publish_date_col)} IS NOT NULL
            '''
        )
        display(publication_lag["publication_lag_days"].describe())
else:
    print("Published 483 table not found.")

## 9. FEI coverage and overlap across sources

In [ ]:
fei_sets: dict[str, set[str]] = {}
fei_coverage_rows = []

for logical_name in TABLES:
    fei_column = pick_column(logical_name, "fei_number", "fei")
    if not fei_column:
        continue
    feis = read_sql(
        f'''
        SELECT DISTINCT btrim({quote_identifier(fei_column)}::text)
            AS fei_number
        FROM {qualified_table(logical_name)}
        WHERE {quote_identifier(fei_column)} IS NOT NULL
          AND btrim({quote_identifier(fei_column)}::text) <> ''
        '''
    )["fei_number"].astype(str)
    fei_sets[logical_name] = set(feis)
    fei_coverage_rows.append(
        {
            "table": logical_name,
            "distinct_feis": len(fei_sets[logical_name]),
        }
    )

display(pd.DataFrame(fei_coverage_rows).sort_values("distinct_feis", ascending=False))

overlap_names = list(fei_sets)
overlap_matrix = pd.DataFrame(
    index=overlap_names,
    columns=overlap_names,
    dtype=int,
)
for left in overlap_names:
    for right in overlap_names:
        overlap_matrix.loc[left, right] = len(
            fei_sets[left] & fei_sets[right]
        )

display(overlap_matrix)

if not overlap_matrix.empty:
    plt.figure(figsize=(8, 6))
    sns.heatmap(overlap_matrix.astype(int), annot=True, fmt="d", cmap="Blues")
    plt.title("Shared FEIs across FDA datasets")
    plt.tight_layout()
    plt.show()

## 10. Build an inspection-only modelling cohort

This block constructs one row per physical inspection and derives historical variables
using only earlier inspections for the same FEI. It does not write to PostgreSQL.

Because the public data does not include the full universe of eligible FEIs, the cohort is
restricted to repeat inspections. Its interpretation is therefore conditional on an FEI
already appearing in FDA inspection data.

In [ ]:
classification_dataset = pd.DataFrame()

required_model_columns = (
    inspection_id_col,
    fei_col,
    classification_col,
    inspection_date_col,
) if "inspections" in TABLES else ()

if required_model_columns and all(required_model_columns):
    inspection_id_sql = quote_identifier(inspection_id_col)
    fei_sql = quote_identifier(fei_col)
    classification_sql = quote_identifier(classification_col)
    inspection_date_sql = quote_identifier(inspection_date_col)

    classification_dataset = read_sql(
        f'''
        WITH physical_inspections AS (
            SELECT
                {inspection_id_sql} AS inspection_id,
                btrim({fei_sql}::text) AS fei_number,
                MIN({inspection_date_sql}) AS prediction_date,
                MAX(
                    CASE upper({classification_sql}::text)
                        WHEN 'NAI' THEN 0
                        WHEN 'VAI' THEN 1
                        WHEN 'OAI' THEN 2
                    END
                ) AS target_severity
            FROM {qualified_table("inspections")}
            WHERE {fei_sql} IS NOT NULL
              AND btrim({fei_sql}::text) <> ''
              AND upper({classification_sql}::text)
                    IN ('NAI', 'VAI', 'OAI')
              AND {inspection_date_sql} IS NOT NULL
            GROUP BY {inspection_id_sql}, btrim({fei_sql}::text)
        ),
        ordered AS (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY fei_number
                    ORDER BY prediction_date, inspection_id
                ) - 1 AS prior_inspection_count,
                LAG(prediction_date) OVER (
                    PARTITION BY fei_number
                    ORDER BY prediction_date, inspection_id
                ) AS previous_inspection_date,
                LAG(target_severity) OVER (
                    PARTITION BY fei_number
                    ORDER BY prediction_date, inspection_id
                ) AS previous_severity,
                COUNT(*) FILTER (WHERE target_severity = 0) OVER (
                    PARTITION BY fei_number
                    ORDER BY prediction_date, inspection_id
                    ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                ) AS prior_nai_count,
                COUNT(*) FILTER (WHERE target_severity = 1) OVER (
                    PARTITION BY fei_number
                    ORDER BY prediction_date, inspection_id
                    ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                ) AS prior_vai_count,
                COUNT(*) FILTER (WHERE target_severity = 2) OVER (
                    PARTITION BY fei_number
                    ORDER BY prediction_date, inspection_id
                    ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                ) AS prior_oai_count
            FROM physical_inspections
        )
        SELECT
            inspection_id,
            fei_number,
            prediction_date,
            CASE target_severity
                WHEN 0 THEN 'NAI'
                WHEN 1 THEN 'VAI'
                WHEN 2 THEN 'OAI'
            END AS target_classification,
            target_severity,
            target_severity > 0 AS target_adverse,
            prior_inspection_count,
            prediction_date - previous_inspection_date
                AS days_since_previous_inspection,
            CASE previous_severity
                WHEN 0 THEN 'NAI'
                WHEN 1 THEN 'VAI'
                WHEN 2 THEN 'OAI'
            END AS previous_classification,
            CASE
                WHEN target.prior_inspection_count = 0 THEN 1
                ELSE 0
            END::SMALLINT AS is_first_observed_inspection,
            COALESCE(prior_nai_count, 0) AS prior_nai_count,
            COALESCE(prior_vai_count, 0) AS prior_vai_count,
            COALESCE(prior_oai_count, 0) AS prior_oai_count,
            (
                COALESCE(prior_vai_count, 0)
                + COALESCE(prior_oai_count, 0)
            )::double precision
            / NULLIF(prior_inspection_count, 0)
                AS prior_adverse_rate
        FROM ordered
        ORDER BY prediction_date, inspection_id
        '''
    )

    display(classification_dataset.head())
    display(classification_dataset.describe(include="all").T)
else:
    print(
        "Could not construct the modelling cohort because one or more "
        "required inspection columns were not found."
    )

In [ ]:
if not classification_dataset.empty:
    target_counts = (
        classification_dataset["target_classification"]
        .value_counts(dropna=False)
        .rename_axis("classification")
        .reset_index(name="count")
    )
    target_counts["rate"] = (
        target_counts["count"] / target_counts["count"].sum()
    )
    display(target_counts)

    plt.figure(figsize=(7, 4))
    sns.barplot(
        data=target_counts,
        x="classification",
        y="count",
        order=["NAI", "VAI", "OAI"],
    )
    plt.title("Classification target distribution: repeat inspections")
    plt.tight_layout()
    plt.show()

    print(
        "Candidate temporal range:",
        classification_dataset["prediction_date"].min(),
        "to",
        classification_dataset["prediction_date"].max(),
    )

## 11. Interpretation checklist

Before creating the production feature table, document:

1. **Grain:** one physical inspection, using the worst project-area classification.
2. **Population:** FEIs with at least one prior inspection.
3. **Prediction time:** preferably inspection start; if only inspection end is available,
   state that it is being used as a proxy.
4. **Leakage prevention:** target-inspection citations, 483s published afterward, recalls
   afterward, and later compliance actions must not be predictors.
5. **Availability time:** use `publish_date` for Published 483 information.
6. **Selection bias:** predictions are conditional on inclusion in FDA inspection data.
7. **Validation:** use a temporal split and report calibration as well as discrimination.

Recommended next step: turn the inspection-only cohort into a versioned SQL feature table,
then add citation, compliance-action, Published 483, and recall aggregates one source at a
time with strict `event_date < prediction_date` conditions.

In [ ]:
# Run this when finished with the notebook session.
engine.dispose()
print("Database connection pool disposed.")

In [ ]:
res = read_sql(
"""
WITH normalized_inspection_rows AS (
    /*
     * Normalize inspection rows and convert the classification
     * into an ordered severity.
     */
    SELECT
        inspection_id,
        btrim(fei_number::text) AS fei_number,
        inspection_end_date,
        NULLIF(btrim(state), '') AS state,
        NULLIF(btrim(product_type), '') AS product_type,
        project_area,

        CASE upper(btrim(classification_code))
            WHEN 'NAI' THEN 0
            WHEN 'VAI' THEN 1
            WHEN 'OAI' THEN 2
        END AS severity

    FROM public.inspections

    WHERE inspection_id IS NOT NULL
      AND fei_number IS NOT NULL
      AND country = 'United States'
      AND btrim(fei_number::text) <> ''
      AND inspection_end_date IS NOT NULL
      AND upper(btrim(classification_code))
            IN ('NAI', 'VAI', 'OAI')
),

inspection_events AS (
    /*
     * Collapse project-area rows into one physical inspection.
     * The worst project-area classification becomes the target.
     */
    SELECT
        inspection_id,
        fei_number,

        MAX(inspection_end_date)
            AS inspection_end_date,

        MAX(severity)
            AS severity,

        MAX(state)
            AS state,

        COUNT(*)::INTEGER
            AS source_row_count,

        COUNT(DISTINCT project_area)::INTEGER
            AS project_area_count

    FROM normalized_inspection_rows

    GROUP BY
        inspection_id,
        fei_number
),

labeled_inspections AS (
    SELECT
        inspection_id,
        fei_number,

        inspection_end_date AS prediction_date,

        state,
        severity,

        CASE severity
            WHEN 0 THEN 'NAI'
            WHEN 1 THEN 'VAI'
            WHEN 2 THEN 'OAI'
        END AS classification,

        severity > 0 AS adverse_classification,

        source_row_count,
        project_area_count

    FROM inspection_events
),

inspection_history AS (
    /*
     * Summarize all strictly earlier inspections and identify
     * the most recent one.
     */
    SELECT
        target.inspection_id,

        COUNT(previous.inspection_id)::INTEGER
            AS prior_inspection_count,

        MIN(previous.prediction_date)
            AS first_inspection_date,

        MAX(previous.prediction_date)
            AS previous_inspection_date,

        MAX(previous.severity)::INTEGER
            AS historical_worst_severity,

        (
            ARRAY_AGG(
                previous.inspection_id
                ORDER BY
                    previous.prediction_date DESC,
                    previous.inspection_id DESC
            ) FILTER (
                WHERE previous.inspection_id IS NOT NULL
            )
        )[1] AS previous_inspection_id,

        (
            ARRAY_AGG(
                previous.severity
                ORDER BY
                    previous.prediction_date DESC,
                    previous.inspection_id DESC
            ) FILTER (
                WHERE previous.inspection_id IS NOT NULL
            )
        )[1]::INTEGER AS previous_severity

    FROM labeled_inspections AS target

    LEFT JOIN labeled_inspections AS previous
        ON previous.fei_number = target.fei_number
       AND previous.prediction_date < target.prediction_date

    GROUP BY target.inspection_id
),

eligible_targets AS (
    /*
     * Restrict the cohort to inspections with at least one
     * observable previous inspection.
     */
    SELECT
        target.*,

        history.prior_inspection_count,
        history.first_inspection_date,
        history.previous_inspection_date,
        history.historical_worst_severity,
        history.previous_inspection_id,
        history.previous_severity

    FROM labeled_inspections AS target

    JOIN inspection_history AS history
        USING (inspection_id)
),

inspection_product_types AS (
    /*
     * Retain each distinct product type associated with a
     * physical inspection.
     */
    SELECT DISTINCT
        inspection_id,
        fei_number,
        product_type

    FROM normalized_inspection_rows

    WHERE product_type IS NOT NULL
),

product_history AS (
    /*
     * Only product types from earlier inspections are included.
     */
    SELECT
        target.inspection_id,

        COUNT(
            DISTINCT products.product_type
        )::INTEGER AS historical_product_type_count,

        MAX(
            CASE
                WHEN lower(products.product_type)
                    LIKE '%food%'
                THEN 1 ELSE 0
            END
        )::SMALLINT AS has_prior_product_food,

        MAX(
            CASE
                WHEN lower(products.product_type)
                    LIKE '%drug%'
                  OR lower(products.product_type)
                    LIKE '%pharma%'
                THEN 1 ELSE 0
            END
        )::SMALLINT AS has_prior_product_drug,

        MAX(
            CASE
                WHEN lower(products.product_type)
                    LIKE '%device%'
                THEN 1 ELSE 0
            END
        )::SMALLINT AS has_prior_product_device,

        MAX(
            CASE
                WHEN lower(products.product_type)
                    LIKE '%biologic%'
                  OR lower(products.product_type)
                    LIKE '%blood%'
                  OR lower(products.product_type)
                    LIKE '%tissue%'
                THEN 1 ELSE 0
            END
        )::SMALLINT AS has_prior_product_biologic,

        MAX(
            CASE
                WHEN lower(products.product_type)
                    LIKE '%veterinar%'
                  OR lower(products.product_type)
                    LIKE '%animal%'
                THEN 1 ELSE 0
            END
        )::SMALLINT AS has_prior_product_veterinary,

        MAX(
            CASE
                WHEN lower(products.product_type)
                    LIKE '%tobacco%'
                THEN 1 ELSE 0
            END
        )::SMALLINT AS has_prior_product_tobacco,

        MAX(
            CASE
                WHEN products.product_type IS NOT NULL
                 AND lower(products.product_type)
                        NOT LIKE '%food%'
                 AND lower(products.product_type)
                        NOT LIKE '%drug%'
                 AND lower(products.product_type)
                        NOT LIKE '%pharma%'
                 AND lower(products.product_type)
                        NOT LIKE '%device%'
                 AND lower(products.product_type)
                        NOT LIKE '%biologic%'
                 AND lower(products.product_type)
                        NOT LIKE '%blood%'
                 AND lower(products.product_type)
                        NOT LIKE '%tissue%'
                 AND lower(products.product_type)
                        NOT LIKE '%veterinar%'
                 AND lower(products.product_type)
                        NOT LIKE '%animal%'
                 AND lower(products.product_type)
                        NOT LIKE '%tobacco%'
                THEN 1 ELSE 0
            END
        )::SMALLINT AS has_prior_product_other

    FROM eligible_targets AS target

    LEFT JOIN labeled_inspections AS previous
        ON previous.fei_number = target.fei_number
       AND previous.prediction_date
            < target.prediction_date

    LEFT JOIN inspection_product_types AS products
        ON products.inspection_id
            = previous.inspection_id
       AND products.fei_number
            = previous.fei_number

    GROUP BY target.inspection_id
),

citation_history AS (
    /*
     * Citations are obtained only through earlier inspections.
     * Citations from the target inspection cannot enter.
     */
    SELECT
        target.inspection_id,

        COUNT(citation.id)::INTEGER
            AS prior_citation_count,

        COUNT(citation.id) FILTER (
            WHERE previous.inspection_id
                = target.previous_inspection_id
        )::INTEGER
            AS previous_inspection_citation_count,

        COUNT(
            DISTINCT citation.inspection_id
        )::INTEGER
            AS prior_inspections_with_citations

    FROM eligible_targets AS target

    LEFT JOIN labeled_inspections AS previous
        ON previous.fei_number = target.fei_number
       AND previous.prediction_date
            < target.prediction_date

    LEFT JOIN public.inspections_citations AS citation
        ON citation.inspection_id
            = previous.inspection_id

    GROUP BY target.inspection_id
),

cfr_history AS (
    /*
     * Count the number of earlier inspections in which each
     * normalized CFR provision appeared.
     */
    SELECT
        target.inspection_id,

        upper(
            btrim(citation.act_cfr_number)
        ) AS normalized_cfr,

        COUNT(
            DISTINCT citation.inspection_id
        )::INTEGER AS cited_inspection_count

    FROM eligible_targets AS target

    JOIN labeled_inspections AS previous
        ON previous.fei_number = target.fei_number
       AND previous.prediction_date
            < target.prediction_date

    JOIN public.inspections_citations AS citation
        ON citation.inspection_id
            = previous.inspection_id

    WHERE citation.act_cfr_number IS NOT NULL
      AND btrim(citation.act_cfr_number) <> ''

    GROUP BY
        target.inspection_id,
        upper(btrim(citation.act_cfr_number))
),

repeated_cfr_history AS (
    /*
     * A repeated CFR is one cited in at least two distinct
     * previous inspections.
     */
    SELECT
        inspection_id,

        COUNT(*) FILTER (
            WHERE cited_inspection_count >= 2
        )::INTEGER AS repeated_cfr_count

    FROM cfr_history

    GROUP BY inspection_id
),

published_483_history AS (
    /*
     * A Published 483 is included only if:
     * 1. its underlying record predates the target; and
     * 2. it was publicly published before the target.
     */
    SELECT
        target.inspection_id,

        COUNT(
            DISTINCT published.record_id
        )::INTEGER AS prior_published_483_count

    FROM eligible_targets AS target

    LEFT JOIN public.published483 AS published
        ON btrim(published.fei_number::text)
            = target.fei_number
       AND published.record_date
            < target.prediction_date
       AND published.publish_date
            < target.prediction_date

    GROUP BY target.inspection_id
),

compliance_action_events AS (
    /*
     * Deduplicate action-product rows into site-level actions.
     */
    SELECT DISTINCT
        btrim(fei_number::text) AS fei_number,
        action_taken_date,
        lower(btrim(action_type)) AS action_type,

        COALESCE(
            NULLIF(
                btrim(case_injunction_id::text),
                ''
            ),
            concat_ws(
                '|',
                btrim(fei_number::text),
                action_taken_date::text,
                lower(btrim(action_type))
            )
        ) AS action_key

    FROM public.compliance_actions

    WHERE fei_number IS NOT NULL
      AND btrim(fei_number::text) <> ''
      AND action_taken_date IS NOT NULL
),

warning_letter_history AS (
    SELECT
        target.inspection_id,

        COUNT(
            DISTINCT action.action_key
        ) FILTER (
            WHERE action.action_type
                LIKE '%warning%'
        )::INTEGER AS prior_warning_letter_count

    FROM eligible_targets AS target

    LEFT JOIN compliance_action_events AS action
        ON action.fei_number
            = target.fei_number
       AND action.action_taken_date
            < target.prediction_date

    GROUP BY target.inspection_id
),

recall_history AS (
    /*
     * event_id is counted rather than product rows because one
     * recall event can contain multiple products.
     */
    SELECT
        target.inspection_id,

        COUNT(
            DISTINCT recall.event_id
        )::INTEGER AS prior_recall_event_count

    FROM eligible_targets AS target

    LEFT JOIN public.recalls AS recall
        ON btrim(recall.fei_number::text)
            = target.fei_number
       AND recall.center_classification_date
            < target.prediction_date

    GROUP BY target.inspection_id
)

SELECT
    target.inspection_id,
    target.fei_number,
    target.prediction_date,

    /* Target variables */
    target.classification
        AS target_classification,

    target.adverse_classification
        AS target_adverse,

    /* Inspection history */
    target.prior_inspection_count,
    CASE
        WHEN target.prior_inspection_count = 0 THEN 1
        ELSE 0
    END::SMALLINT AS is_first_observed_inspection,

    COALESCE(
        target.prediction_date - target.previous_inspection_date, 0
    )
        AS days_since_previous_inspection,

    CASE
        WHEN target.previous_inspection_id IS NULL THEN 0
        WHEN target.previous_severity = 0 THEN 0
        WHEN target.previous_severity IN (1, 2) THEN 1
    END::SMALLINT
        AS previous_classification_adverse,

    /* Historical product types */
    COALESCE(
        product.historical_product_type_count,
        0
    ) AS historical_product_type_count,

    COALESCE(
        product.has_prior_product_food,
        0
    ) AS has_prior_product_food,

    COALESCE(
        product.has_prior_product_drug,
        0
    ) AS has_prior_product_drug,

    COALESCE(
        product.has_prior_product_device,
        0
    ) AS has_prior_product_device,

    COALESCE(
        product.has_prior_product_biologic,
        0
    ) AS has_prior_product_biologic,

    COALESCE(
        product.has_prior_product_veterinary,
        0
    ) AS has_prior_product_veterinary,

    COALESCE(
        product.has_prior_product_tobacco,
        0
    ) AS has_prior_product_tobacco,

    COALESCE(
        product.has_prior_product_other,
        0
    ) AS has_prior_product_other,

    /* Historical citations */
    COALESCE(
        citation.prior_citation_count,
        0
    ) AS prior_citation_count,

    COALESCE(
        citation.previous_inspection_citation_count,
        0
    ) AS previous_inspection_citation_count,

    CASE
        WHEN target.prior_inspection_count = 0
            THEN 0.0
        ELSE
            COALESCE(
            citation.prior_citation_count,
            0
        )::DOUBLE PRECISION
        / target.prior_inspection_count
    END AS prior_citations_per_inspection,

    COALESCE(
        repeated_cfr.repeated_cfr_count,
        0
    ) AS repeated_cfr_count,

    /* Facility geography */
    target.state,

    /* Published Form 483 history */
    COALESCE(
        published_483.prior_published_483_count,
        0
    ) AS prior_published_483_count,

    CASE
        WHEN target.prior_inspection_count = 0
            THEN 0.0

        ELSE
            COALESCE(
                published_483.prior_published_483_count,
                0
            )::DOUBLE PRECISION
            / target.prior_inspection_count
    END AS prior_published_483_count_per_inspection,

    /* Warning-letter history */
    COALESCE(
        warning.prior_warning_letter_count,
        0
    ) AS prior_warning_letter_count,

    COALESCE(
        warning.prior_warning_letter_count,
        0
    )::DOUBLE PRECISION
        / NULLIF(
            target.prior_inspection_count,
            0
        ) AS prior_warning_letter_count_per_inspection,

    /* Recall history */
    COALESCE(
        recall.prior_recall_event_count,
        0
    ) AS prior_recall_event_count

FROM eligible_targets AS target

LEFT JOIN product_history AS product
    USING (inspection_id)

LEFT JOIN citation_history AS citation
    USING (inspection_id)

LEFT JOIN repeated_cfr_history AS repeated_cfr
    USING (inspection_id)

LEFT JOIN published_483_history AS published_483
    USING (inspection_id)

LEFT JOIN warning_letter_history AS warning
    USING (inspection_id)

LEFT JOIN recall_history AS recall
    USING (inspection_id);
"""
)
res.sample(10)

In [ ]:
res["prior_inspection_count"].value_counts()